## 1. Introducing Tool Use
### (1) Why need tools?
Tools allow Claude to access information from the outside world, extending its capabilities beyond what it learned during training. 
By default, Claude only knows information from its training data and can't access current events, real-time data, or external systems.

### (2) How toolUse works?
Tool use follows a specific back-and-forth pattern between your application and Claude. Here's the complete flow:
- Initial Request: You send Claude a question along with instructions on how to get extra data from external sources
- Tool Request: Claude analyzes the question and decides it needs additional information, then asks for specific details about what data it needs
- Data Retrieval: Your server runs code to fetch the requested information from external APIs or databases
- Final Response: You send the retrieved data back to Claude, which then generates a complete response using both the original question and the fresh data

### (3) Overview
- step1: write a tool function 
- step2: write a JSON schema
- step3: call Claude with JSON schema
- step4: run tool
- step4: add tool result and call Claude again

## 2. Tool function
### (1) What are tools
Tools re Python functions that Claude can call when it needs additional data to help users.

### (2) Best Pratices for tool functions
- Use descriptive names: Both your function name and parameter names should clearly indicate their purpose
- Validate inputs: Check that required parameters aren't empty or invalid, and raise errors when they are
- Provide meaningful error messages: Claude can see error messages and might retry the function call with corrected parameters

### (3) Implementation - building your first tool function



## 3. Tool JSON schema
### (1) why need JSON schema
JSON schema tells Claude what arguements your function requires

### (2) Best Practices
- Explain what the tool does, when to use it, and what it returns
- Aim for 3-4 sentences
- Provide detailed descriptions for each argument
- 3 main parts for a complete tool specification
    - name
    - description
    - input_schema
- !! Use the pattern of function_name followed by function_name_schema to keep your schemas organized and easy to match with their corresponding functions.

### (3) The easiest way to generaate schemas
Instead of writing JSON schemas from scratch, you can use Claude itself to generate them. Here's the process:

Copy your tool function code
Go to Claude and ask it to write a JSON schema for tool calling
Include the Anthropic documentation on tool use as context
Let Claude generate a properly formatted schema following best practices
The prompt should be something like: "Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation."

### (4) ToolPAram --> help check typos for JSON schema


## 4. Call Claude with JSON schema 
### (1) Making tool-enabled API calls
#### 1) Implementation --> include a tools paramter (a list of JSON schemas)
#### 2) understanding multi-block message
A multi-block message typically contains:

Text Block - Human-readable text explaining what Claude is doing (like "I can help you find out the current time. Let me find that information for you")
ToolUse Block - Instructions for your code about which tool to call and what parameters to use
The ToolUse block includes:

An ID for tracking the tool call
The name of the function to call (like "get_current_datetime")
Input parameters formatted as a dictionary
The type designation "tool_use" 

#### 3) append the ToolUseMessage into historical messages
#### 4) The Complete Tool Usage Flow -- implementation


## 5. Run tool Sending tool results 
### (1) Run the tool function -- how to get input from toolUseBlock
### (2) Build ToolResultBlock ---> "tool_use_ud", "content", "is_error"
### (3) Building the follow-up request with the ToolResultBlock


## 6. Multi-turn conversations with tools
Complete Workflow
The complete multi-turn conversation works like this:

Send user message to Claude with available tools
Claude responds with text and/or tool requests
Execute all requested tools and create result blocks
Send tool results back as a user message
Repeat until Claude provides a final answer
### (1) Building a conversation loop ---> ps: need to upgrade add_user_message, add_assistant_message, and chat functions correspondingly to handle all types of content
### (2) Handling multiple tool calls
### (3) Tool Result Blocks + Error Handling
### (4) Scalable Tool Routing

## 7. Running multiple tools
The Simple Pattern for Adding Tools
Once you have the core tool infrastructure, adding new tools follows this pattern:

1. Create the tool function implementation
2. Define the tool schema
3. Add the schema to the tools list in run_conversation
4. Add a case for the tool in run_tool


## 8. Fine grained Tool calling
### (1) Tool streaming --> fine_grained=True to your API call
### (2) Text Edit Tool --> see details in 09_advanced_tool_TextEditorTool.ipynb
### (3) 


## Q&A 
### (1) Compare tool_use_block vs tool_result_block
ToolUseBlock(
    id='toolu_01QtKMDNkU1FBj73NP2FbF8w',
    input={'date_format': '%H:%M:%S'},
    name='get_current_datetime',
    type='tool_use'
)

tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": json.dumps(tool_output),
    "is_error": False
}

In [1]:
from datetime import datetime 

In [2]:
## 2. Tool function

### building your first tool function
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime()  # 2026-05-13 20:18:29

'2026-05-13 22:02:45'

In [3]:
# Just hour and minute: "20:19"
get_current_datetime("%H:%M")

'22:02'

In [4]:
## 3. Tool Schema
get_current_datetime_schema = {
  "name": "get_current_datetime",
  "description": "Returns the current date and time as a formatted string based on the user's system clock. Use this tool whenever you need to know the present moment — for example, to timestamp an event, compute a relative time ('how long until...'), or include the current date in a response. Do not use it for historical dates, future dates, or times in other time zones; it only reports the local 'now'. Returns a single string formatted according to the `date_format` argument (defaults to ISO-like 'YYYY-MM-DD HH:MM:SS').",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string controlling how the current datetime is rendered. Must be a non-empty string composed of strftime directives (e.g., '%Y' for 4-digit year, '%m' for month, '%d' for day, '%H' for 24-hour hour, '%M' for minute, '%S' for second) and any literal separators. Examples: '%Y-%m-%d %H:%M:%S' → '2026-05-13 14:30:45'; '%B %d, %Y' → 'May 13, 2026'; '%H:%M' → '14:30'. Defaults to '%Y-%m-%d %H:%M:%S' if omitted. An empty string will cause the tool to raise an error.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
}

In [5]:
### ToolPAram --> help check typos for JSON schema
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time as a formatted string based on the user's system clock. Use this tool whenever you need to know the present moment — for example, to timestamp an event, compute a relative time ('how long until...'), or include the current date in a response. Do not use it for historical dates, future dates, or times in other time zones; it only reports the local 'now'. Returns a single string formatted according to the `date_format` argument (defaults to ISO-like 'YYYY-MM-DD HH:MM:SS').",
        "input_schema": {
            "type": "object",
            "properties": {
            "date_format": {
                "type": "string",
                "description": "A Python strftime format string controlling how the current datetime is rendered. Must be a non-empty string composed of strftime directives (e.g., '%Y' for 4-digit year, '%m' for month, '%d' for day, '%H' for 24-hour hour, '%M' for minute, '%S' for second) and any literal separators. Examples: '%Y-%m-%d %H:%M:%S' → '2026-05-13 14:30:45'; '%B %d, %Y' → 'May 13, 2026'; '%H:%M' → '14:30'. Defaults to '%Y-%m-%d %H:%M:%S' if omitted. An empty string will cause the tool to raise an error.",
                "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
    }
)

In [6]:
## 4. Call Claude with JSON schema 
### (1) Making tool-enabled API calls
#### 1) Implementation --> include a tools paramter (a list of JSON schemas)
from anthropic import Anthropic
client = Anthropic()
messages = []
model = "claude-haiku-4-5-20251001"
messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model = model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [7]:
#### 2) understanding multi-block message
response.content

[ToolUseBlock(id='toolu_01T3PJpmAVsvwqc7J8SYjDYM', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]

In [8]:
len(response.content)

1

In [9]:
#### 3) append the ToolUseMessage into historical messages
messages.append(
    {
        "role": "assistant",
        "content": response.content
    }
)

In [10]:
### 4) The Complete Tool Usage Flow
def add_user_message(messages, user_content):
    new_message = {"role":"user", "content": user_content}
    messages.append(new_message)

def add_assistant_message(messages, assistant_content):
    new_message = {"role":"assistant", "content": assistant_content}
    messages.append(new_message)

def chat(model, max_tokens, messages, system=None, temperature=1.0, tools = []):
    params = {
        "model": model, 
        "max_tokens": max_tokens,
        "messages": messages
    }

    if system:
        params["system"] = system
    if temperature:
        params["temperature"] = temperature  
    if tools:
        params["tools"] = tools
    
    response = client.messages.create(**params)
    return response

In [11]:
response.content[0]

ToolUseBlock(id='toolu_01T3PJpmAVsvwqc7J8SYjDYM', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')

In [12]:
## 5. Run tool Sending tool results 
### (1) Run the tool function -- how to get input from toolUseBlock
get_current_datetime(**response.content[0].input)

'22:05:34'

In [13]:
### (2) Building the ToolResultBlock "tool_use_ud", "content", "is_error"
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": "15:04:22",
        "is_error": False
    }]
})

In [14]:
### (3) Building the follow-up request with the ToolResultBlock
### (4) Making the final request
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_01PwCisnBpTAhRXubgQGmQ2u', container=None, content=[TextBlock(citations=None, text='The exact time is **15:04:22** (3:04:22 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=976, output_tokens=23, server_tool_use=None, service_tier='standard'))

In [15]:
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01T3PJpmAVsvwqc7J8SYjDYM', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01T3PJpmAVsvwqc7J8SYjDYM',
    'content': '15:04:22',
    'is_error': False}]}]

In [ ]:
## 6. Multi-turn conversations with tools
### (1) Building a conversation loop
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
        # This allows you to pass in either a string, a list of blocks, or a complete message object.
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    user_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
        # This allows you to pass in either a string, a list of blocks, or a complete message object.
    }
    messages.append(user_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    
    if tools:
        params["tools"] = tools
        
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message

# Extracting Text from Messages
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

def run_conversation(messages, model="claude-haiku-4-5-20251001", max_tokens=1000, tools=[]):
    while True:
        response = chat(model=model, max_tokens=1000, messages=messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages=messages, assistant_content=response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [ ]:
### (2) Handling multiple tool calls
def run_tools(message):
    tool_requests = [
        block for block in message.content if tool_request.type == "too_use"
    ]

    tool_result_block = []
    for tool_request in tool_requests:
        # Process each tool request..
        run_tool()
        pass

In [ ]:
### (3) Tool Result Blocks + Error Handling
# tool result block
tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": json.dumps(tool_output),
    "is_error": False
}

# error handling 
try:
    tool_output = run_tool(tool_request.name, tool_request.input)
    tool_result_block = {
        "type": "tool_result",
        "tool_use_id": tool_request.id,
        "content": json.dumps(tool_output),
        "is_error": False
    }
except Exception as e:
    tool_result_block = {
        "type": "tool_result", 
        "tool_use_id": tool_request.id,
        "content": f"Error: {e}",
        "is_error": True
    }

In [ ]:
### (4) Scalable Tool Routing
def run_tools(message):
    tool_requests = [
        block for block in message.content if tool_request.type == "too_use"
    ]

    tool_result_blocks = []
    for tool_request in tool_requests:
        tool_output = run_tool(tool_name, tool_input)
        tool_result_block = {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_output),
            "is_error": False
        }
        tool_result_blocks.append(tool_result_block)
    
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "another_tool":
        return another_tool(**tool_input)
    # Add more tools as needed